# 初始化客户端


In [1]:
# 初始化客户端
from pymilvus import MilvusClient
from rich import print as rprint

client = MilvusClient(uri="http://localhost:19530")
database_name = "rag"
# 创建数据库（必须是不存在）
exist_databases = client.list_databases()
if database_name not in exist_databases:
    client.create_database(database_name)

# 使用数据库
client.use_database(database_name)

In [2]:
# 创建 collection
collection_name = "doc_test"
DIM = 768

if client.has_collection(collection_name):
    client.drop_collection(collection_name)
else :
    client.create_collection(
        collection_name=collection_name,
        dimension=DIM,
        metric_type="COSINE",
    )

In [3]:
# 初始化嵌入模型
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

# 使用本地 Ollama 服务
embedding_model = OllamaEmbeddings(
    model="embeddinggemma:latest",  # 嵌入模型名称
    base_url="http://localhost:11434"  # Ollama 服务地址
)

In [4]:
# 读取文档并且切分
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

document_path = "../asset/load/news.txt"

# 加载文档
loader = TextLoader(file_path=document_path, encoding="utf-8")
documents = loader.load()

# 切分文档

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=80,separators=[
    "\n\n", "\n", " ", ""
])

document_chunks = splitter.split_documents(documents)

print(len(document_chunks))

for doc in document_chunks:
    print("-" * 20)
    print(doc.page_content)




25
--------------------
## 关于举办第十五届“中国软件杯”大学生软件设计大赛的通知

各级教育主管部门、各院校：
--------------------
第十五届"中国软件杯"大学生软件设计大赛(简称"大赛")报名工作已于近期启动。今年全国两会期间，习近平总书记在参加他所在的十四届全国人大三次会议江苏代表团审议时强调，抓科技创新，要着眼建设现代化产业体系，坚持教育、科技、人才一起抓，既多出科技成果，又把科技成果转化为实实在在的生产力。“中国软件杯”大赛是加强产教融合、科教融汇的桥梁纽带，是教育、科技、人才“三位一体”协同、融合发展的重要力量。继续
--------------------
科技成果，又把科技成果转化为实实在在的生产力。“中国软件杯”大赛是加强产教融合、科教融汇的桥梁纽带，是教育、科技、人才“三位一体”协同、融合发展的重要力量。继续办好第十五届“中国软件杯”大赛，将为推动我国软件产业高质量发展、发展新质生产力贡献力量。现将第十五届大赛有关事项通知如下：
--------------------
## 一、大赛目的

大赛旨在落实《特色化示范性软件学院建设指南(试行)》和《工业和信息化部关于加强和改进工业和信息化人才队伍建设的实施意见》，推动实施科教兴国战略、人才强国战略和创新驱动发展战略，科学引导高校学子参加科研活动，探索产教融合育人路径，推动软件产业高质量发展。
--------------------
通过搭建富有自由、开放、创新精神的软件设计大赛平台，加强高校人才培养和新兴信息产业需求的有效衔接，推动产教深度融合，加快培育更多高端、优秀软件人才，推动关键核心技术突破，增强产业自主创新能力，实现产业高质量发展。

## 二、参赛对象及规则

(一) 参赛对象:
--------------------
## 二、参赛对象及规则

(一) 参赛对象:

全日制普通高等院校(含海外院校)在籍学生【含本科生、研究生及以上学历、中职、高职(高专)、职业本科】，报名将分两组进行，其中本科生、研究生及以上学历、中职、高职(高专)、职业本科可报A组赛题，中职、高职(高专)、职业本科可报B组赛题。

(二) 参赛形式:
--------------------
以组队报名形式参赛，每队成员不超过4名（含4名，其中队长1名，指导教师1名，

C:\Users\Administrator\AppData\Local\Temp\ipykernel_10676\2722435293.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
# 生成向量写入Milvus

text = [
    chunk.page_content for chunk in document_chunks
]

print(f"共 {len(text)} 个文档片段")

# 使用 ollama 批量嵌入
vectors = embedding_model.embed_documents(text)

data = [
    {   
        "id": i,
        "text": text[i],
        "vector": vectors[i],
        "source": document_path,
        "chunk_id": i
    }
    for i in range(len(text))
]

# 插入数据
insert_res = client.upsert(
    collection_name=collection_name,
    data=data
)

print("插入结果:", insert_res)

# flush
client.flush(collection_name)

# 统计集合信息
stats = client.get_collection_stats(collection_name)
print("集合统计:", stats)

共 25 个文档片段
插入结果: {'upsert_count': 25, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]}
集合统计: {'row_count': 25}


# 创建agent

In [6]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from rich import print as rprint
import os
import dotenv

dotenv.load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
    你是一个助手，你需要根据用户的输入结合检索到的上下文，对用户进行回答。
    """,
)

# 检索

In [8]:
def ask(query:str,limit:int =5):

    query_vector = embedding_model.embed_query(query)

    # 检索向量数据库中数据
    res = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=limit,
        output_fields=["chunk_id","text","source"],
    )

    rprint(res)
    return res[0]

In [9]:
def generate_response(query: str):
    """基于检索结果生成回答"""
    hits = ask(query)
    rprint(f"[yellow]检索到 {len(hits)} 条结果:[/yellow]")
    
    # 格式化上下文 - 将检索到的文档片段拼接
    context_parts = []
    for i, hit in enumerate(hits):
        score = hit.get("distance", 0)
        text = hit.get("entity", {}).get("text", "")
        context_parts.append(f"[片段{i+1}] (相似度: {score:.4f})\n{text}")
    
    context = "\n\n".join(context_parts)
    
    # 构建 prompt
    messages = [
        SystemMessage(content="你是一个助手，你需要根据用户的输入结合检索到的上下文，对用户进行回答。"),
        HumanMessage(content=f"""基于以下上下文回答用户问题。如果上下文不相关，请说明无法从提供的资料中找到答案。

## 上下文
{context}

## 用户问题
{query}""")
    ]
    
    # 直接调用 LLM 生成回答
    response = llm.invoke(messages)
    
    return {
        "answer": response.content,
        "sources": [hit.get("entity", {}).get("source", "") for hit in hits],
        "context": context
    }

In [10]:
# 测试 RAG 问答
query = """第十五届"中国软件杯"大学生软件设计大赛的目的是什么？"""
result = generate_response(query)

print("问题:", query)
print("\n回答:", result["answer"])
print("\n来源:", result["sources"])

data: [[{'id': 0, 'distance': 0.7641327381134033, 'entity': {'text': '## 
关于举办第十五届“中国软件杯”大学生软件设计大赛的通知\n\n各级教育主管部门、各院校：', 'source': 
'../asset/load/news.txt', 'chunk_id': 0}}, {'id': 2, 'distance': 0.7396680116653442, 'entity': {'text': 
'科技成果，又把科技成果转化为实实在在的生产力。“中国软件杯”大赛是加强产教融合、科教融汇的桥梁纽带，是教育、科技、人
才“三位一体”协同、融合发展的重要力量。继续办好第十五届“中国软件杯”大赛，将为推动我国软件产业高质量发展、发展新质生
产力贡献力量。现将第十五届大赛有关事项通知如下：', 'source': '../asset/load/news.txt', 'chunk_id': 2}}, {'id': 4, 
'distance': 0.6945418119430542, 'entity': {'text': 
'通过搭建富有自由、开放、创新精神的软件设计大赛平台，加强高校人才培养和新兴信息产业需求的有效衔接，推动产教深度融合
，加快培育更多高端、优秀软件人才，推动关键核心技术突破，增强产业自主创新能力，实现产业高质量发展。\n\n## 
二、参赛对象及规则\n\n(一) 参赛对象:', 'source': '../asset/load/news.txt', 'chunk_id': 4}}, {'id': 24, 'distance': 
0.6868767142295837, 'entity': {'text': 
'参赛学生群：141102123\n\n大赛微信公众号：2731018448\n\n“中国软件杯”大学生软件设计，组委会\n\n![](images/973a90ea75
9711d2a754f850d270cf319d77781045a880fea8c23f326d9727f0.jpg)', 'source': '../asset/load/news.txt', 'chunk_id': 24}},
{'id': 3, 'distance': 0.6592922210693359, 'entity': {'text': '## 
一、大赛目的\n\n大赛旨在落实《特色化示范性软件学院建设指南(试行)》和《工业和信息化部关于加强和改进工业和信息化人才
队伍建设的实施意见》，推动实施科教兴国战略、人才强国战略和创新驱动发展战略，科学引导高校学子参加科研活动，探索产教
融合育人路径，推动软件产业高质量发展。', 'source': '../asset/load/news.txt', 'chunk_id': 3}}]]

检索到 5 条结果:

问题: 第十五届"中国软件杯"大学生软件设计大赛的目的是什么？

回答: 第十五届“中国软件杯”大学生软件设计大赛的目的主要包括：  
1. **落实国家政策**：贯彻落实《特色化示范性软件学院建设指南(试行)》和《工业和信息化部关于加强和改进工业和信息化人才队伍建设的实施意见》。  
2. **推动战略实施**：推动实施科教兴国战略、人才强国战略和创新驱动发展战略。  
3. **引导学生参与科研**：科学引导高校学子参加科研活动。  
4. **深化产教融合**：探索产教融合育人路径，加强高校人才培养与新兴信息产业需求的衔接。  
5. **助力产业发展**：推动软件产业高质量发展，增强产业自主创新能力，加快培育高端软件人才，为发展新质生产力贡献力量。  
6. **促进协同创新**：大赛作为教育、科技、人才“三位一体”协同发展的重要平台，旨在促进产教融合与科教融汇。

来源: ['../asset/load/news.txt', '../asset/load/news.txt', '../asset/load/news.txt', '../asset/load/news.txt', '../asset/load/news.txt']
